In [1]:
# notebook — cellule 1 : composition des configs v2 et v3
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

from src.config import PROJ_ROOT

CONF_DIR = str(PROJ_ROOT / "conf")
EXPERIMENTS = {"v2_multi": "train/train_sage_gru_2_cat",
               "v3_multi_novalid": "train/train_sage_gru_v3"}


def load_cfg(experiment: str, seed: int, spec_name: str):
    GlobalHydra.instance().clear()
    with initialize_config_dir(config_dir=CONF_DIR, version_base="1.3"):
        cfg = compose(config_name="train", overrides=[f"+experiment={experiment}"])
    # ++ via OmegaConf plutôt que via les overrides : insensible au mode struct
    OmegaConf.set_struct(cfg, False)
    cfg.training.model.seed = seed
    cfg.training.model.feature_spec = spec_name
    OmegaConf.set_struct(cfg, True)
    return cfg


cfg_check = load_cfg(EXPERIMENTS["v3_multi_novalid"], seed=0, spec_name="v3_multi_novalid")
print(OmegaConf.to_yaml(cfg_check.training.model))

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-24 15:28:29.143 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


name: sage_gru_1
method: training
params_model:
  n_features: 6
  hidden: 32
  n_layers: 1
params_training:
  epochs: 30
  lr: 0.001
batch_size: 64
feature_spec: v3_multi_novalid
feature_version:
  segmentation: 2
  subj_category:
  - Assertion
  - Expressive
  - Question
  - Suggestion
seed: 0



In [2]:
# notebook — cellule 2 : sanity check des specs, sans entraînement
from src.detector.subjectivity_aware_generation_estimator.features.feature_spec import get_spec
from src.detector.subjectivity_aware_generation_estimator.training.gru_train import prepare_data

for name in ["v2_multi", "v3_multi_novalid"]:
    spec = get_spec(name)
    print(f"{name:20s} input_dim={spec.input_dim}  hash={spec.hash()}  "
          f"colonnes={list(spec.column_names)}")
    print(f"{'':20s} clé segmentation = {spec.segmentation_key()}")

d = prepare_data(load_cfg(EXPERIMENTS["v3_multi_novalid"], 0, "v3_multi_novalid"))
print(f"\ntrain : {d['Xtr'].shape}  val : {d['Xva'].shape}")
print(f"relectures train {len(d['Ytr'])}  val {len(d['Yva'])}  "
      f"part MGT val {d['Yva'].float().mean():.3f}")
print(f"longueurs val : médiane {d['Lva'].median().item():.0f}  max {d['Lva'].max().item()}")

/data2/cyaiche/cache_dir
v2_multi             input_dim=7  hash=014e6f83506e  colonnes=['Assertion', 'Expressive', 'Neither', 'Question', 'Suggestion', 'subj_z', 'subj_valid']
                     clé segmentation = {'segmentation': 2, 'subj_category': ['Assertion', 'Expressive', 'Question', 'Suggestion']}
v3_multi_novalid     input_dim=6  hash=77627d888210  colonnes=['Assertion', 'Expressive', 'Neither', 'Question', 'Suggestion', 'subj_z']
                     clé segmentation = {'segmentation': 2, 'subj_category': ['Assertion', 'Expressive', 'Question', 'Suggestion']}
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 10000, 'split': 'calibration', 'with_pdf': False, 'seed': 0}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_4fbe88291ebaf69cc7d4bd78214b8d5

In [3]:
# notebook — cellule 3 : run de contrôle v2 vs v3, 3 seeds
import pandas as pd
import numpy as np
from src.config import RESULT_DIR
from pathlib import Path

from src.detector.subjectivity_aware_generation_estimator.training.gru_train import train_gru
from src.detector.subjectivity_aware_generation_estimator.xai.metrics import detection_metrics
from src.detector.subjectivity_aware_generation_estimator.xai.reporting import print_table

SEEDS = (0, 1, 2)
rows, artifacts = [], {}

for spec_name, experiment in EXPERIMENTS.items():
    for seed in SEEDS:
        cfg = load_cfg(experiment, seed=seed, spec_name=spec_name)
        art = train_gru(cfg)
        artifacts[(spec_name, seed)] = art

        res = np.load(Path(RESULT_DIR) / art["hash"] / "gru_result.npz", allow_pickle=True)
        row = {"spec": spec_name, "seed": seed,
               "input_dim": art["feature_spec"]["input_dim"],
               "params": sum(v.numel() for v in art["state_dict"].values()),
               "hash": art["hash"]}
        row.update(detection_metrics(res["y_true"], res["y_proba"], threshold=0.5))
        rows.append(row)
        print(f"--> {spec_name} seed {seed} : AUC {row['auc']:.4f}  "
              f"TPR@1%FPR {row['tpr@fpr1']:.4f}\n")

runs_spec = pd.DataFrame(rows)
print_table(runs_spec[["spec", "seed", "input_dim", "params", "auc", "acc",
                       "tpr@fpr1", "tpr@fpr0_1"]],
            title="Runs individuels", index=False)

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_2
    method: training
    params_model:
      n_features: 7
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
      - Expressive
      - Question
    

Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 10000, 'split': 'calibration', 'with_pdf': False, 'seed': 0}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_4fbe88291ebaf69cc7d4bd78214b8d51.pt
📦 Loading data from cache: /data2/cyaiche/cache_dir/results/implicite_segmentation/4fbe88291ebaf69cc7d4bd78214b8d51_eaf4e799/sentences_degrees_final.csv
appariement : 3017 paires | HWT écartés 0 | versions MGT écartées 1983
générateurs retenus : {'llama': 638, 'qwen': 627, 'gemini': 613, 'claude': 580, 'gpt4o': 559}
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_2
    method: training
    params_model:
      n_features: 7
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
      - Expressive
      - Question
    


              precision    recall  f1-score   support

           0      0.927     0.833     0.877      4468
           1      0.848     0.934     0.889      4468

    accuracy                          0.884      8936
   macro avg      0.887     0.884     0.883      8936
weighted avg      0.887     0.884     0.883      8936

AUC globale : 0.9569

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.930818
      ICLR  2020  790 0.945855
   NeurIPS  2018  802 0.949801
   NeurIPS  2017  732 0.950514
   NeurIPS  2019  802 0.952077
      ICLR  2017  730 0.953357
      ICLR  2019  768 0.955519
      ICLR  2018  796 0.956712
   NeurIPS  2021 1070 0.981176
      ICLR  2021  976 0.987604

AUC par générateur :
 generator    n      auc
    llama 1767 0.915314
   gemini 1751 0.950588
   claude 1780 0.967900
     qwen 1853 0.968724
    gpt4o 1785 0.980342

 /data2/cyaiche/cache_dir/results/60e6785e963e63339c5cdc7343ab34b5/artifact.joblib
--> v2_multi seed 0 : AUC 0.9569  TPR@1

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_2
    method: training
    params_model:
      n_features: 7
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
      - Expressive
      - Question
    


              precision    recall  f1-score   support

           0      0.924     0.838     0.879      4468
           1      0.852     0.931     0.889      4468

    accuracy                          0.884      8936
   macro avg      0.888     0.884     0.884      8936
weighted avg      0.888     0.884     0.884      8936

AUC globale : 0.9578

AUC par cellule :
 conference  year    n      auc
      ICLR  2020  790 0.934196
   NeurIPS  2016 1470 0.934451
   NeurIPS  2018  802 0.951997
      ICLR  2017  730 0.953267
   NeurIPS  2019  802 0.954130
   NeurIPS  2017  732 0.956971
      ICLR  2019  768 0.960341
      ICLR  2018  796 0.968227
   NeurIPS  2021 1070 0.981336
      ICLR  2021  976 0.986848

AUC par générateur :
 generator    n      auc
    llama 1767 0.916436
   gemini 1751 0.953866
   claude 1780 0.969372
     qwen 1853 0.969797
    gpt4o 1785 0.977938

 /data2/cyaiche/cache_dir/results/ac61c4929414a56d9e08365eccf625b4/artifact.joblib
--> v2_multi seed 1 : AUC 0.9578  TPR@1

... (2 outputs left out)

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_multi_novalid
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
     


              precision    recall  f1-score   support

           0      0.891     0.867     0.879      4468
           1      0.870     0.894     0.882      4468

    accuracy                          0.881      8936
   macro avg      0.881     0.881     0.881      8936
weighted avg      0.881     0.881     0.881      8936

AUC globale : 0.9549

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.929598
      ICLR  2020  790 0.936619
   NeurIPS  2018  802 0.942612
   NeurIPS  2017  732 0.945997
   NeurIPS  2019  802 0.948589
      ICLR  2017  730 0.948801
      ICLR  2019  768 0.960381
      ICLR  2018  796 0.962242
   NeurIPS  2021 1070 0.981364
      ICLR  2021  976 0.986601

AUC par générateur :
 generator    n      auc
    llama 1767 0.922292
   gemini 1751 0.952415
     qwen 1853 0.961544
   claude 1780 0.964478
    gpt4o 1785 0.973316

 /data2/cyaiche/cache_dir/results/40267e07f7acff6aea5cee99ae7abf31/artifact.joblib
--> v3_multi_novalid seed 0 : AUC 0.954

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_multi_novalid
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
     


              precision    recall  f1-score   support

           0      0.931     0.842     0.884      4468
           1      0.856     0.937     0.895      4468

    accuracy                          0.890      8936
   macro avg      0.893     0.890     0.889      8936
weighted avg      0.893     0.890     0.889      8936

AUC globale : 0.9611

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.938509
      ICLR  2020  790 0.942189
   NeurIPS  2018  802 0.954627
      ICLR  2017  730 0.958003
   NeurIPS  2019  802 0.958390
   NeurIPS  2017  732 0.958845
      ICLR  2019  768 0.961005
      ICLR  2018  796 0.966251
   NeurIPS  2021 1070 0.982814
      ICLR  2021  976 0.990380

AUC par générateur :
 generator    n      auc
    llama 1767 0.922584
   gemini 1751 0.959891
     qwen 1853 0.969469
   claude 1780 0.973631
    gpt4o 1785 0.979826

 /data2/cyaiche/cache_dir/results/1ba399eda7c9218acf7ff6f7ee40770b/artifact.joblib
--> v3_multi_novalid seed 1 : AUC 0.961

In [4]:
# notebook — cellule 4 : comparaison v2 vs v3
import numpy as np

metric_cols = ["auc", "acc", "tpr@fpr1", "tpr@fpr0_1"]
agg = runs_spec.groupby("spec")[metric_cols].agg(["mean", "std"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg["n"] = runs_spec.groupby("spec").size()
print_table(agg, title="Agrégé sur 3 seeds")

v2, v3 = agg.loc["v2_multi"], agg.loc["v3_multi_novalid"]
for m in metric_cols:
    delta = v3[f"{m}_mean"] - v2[f"{m}_mean"]
    se = np.sqrt(v3[f"{m}_std"] ** 2 / v3["n"] + v2[f"{m}_std"] ** 2 / v2["n"])
    verdict = "indistinguable" if abs(delta) < 2 * se else "ÉCART SIGNIFICATIF"
    print(f"{m:12s}  v3 - v2 = {delta:+.4f}  (± {se:.4f})  -> {verdict}")

== Agrégé sur 3 seeds ==
                  auc_mean  auc_std  acc_mean  acc_std  tpr@fpr1_mean  tpr@fpr1_std  tpr@fpr0_1_mean  tpr@fpr0_1_std  n
spec                                                                                                                   
v2_multi            0.9544   0.0052    0.8773   0.0115         0.4600        0.0181           0.1732          0.0609  3
v3_multi_novalid    0.9569   0.0036    0.8837   0.0051         0.4650        0.0123           0.1673          0.0484  3

auc           v3 - v2 = +0.0026  (± 0.0037)  -> indistinguable
acc           v3 - v2 = +0.0064  (± 0.0072)  -> indistinguable
tpr@fpr1      v3 - v2 = +0.0050  (± 0.0127)  -> indistinguable
tpr@fpr0_1    v3 - v2 = -0.0059  (± 0.0449)  -> indistinguable


In [3]:
# notebook — cellule 5 : grille de fusion sur v3, 5 conditions × 5 seeds
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import RESULT_DIR
from src.detector.subjectivity_aware_generation_estimator.features.feature_spec import SPECS, get_spec
from src.detector.subjectivity_aware_generation_estimator.training.gru_train import train_gru
from src.detector.subjectivity_aware_generation_estimator.xai.metrics import detection_metrics
from src.detector.subjectivity_aware_generation_estimator.xai.reporting import print_table

BASE_EXPERIMENT = EXPERIMENTS["v3_multi_novalid"]
MERGE_SPECS = ["v3_multi_novalid"] + sorted(s for s in SPECS if s.startswith("v3_merge_"))
SEEDS = (0, 1, 2, 3, 4)

# contrôle : toutes les specs doivent partager la clé de segmentation, sinon resegmentation
keys = {s: get_spec(s).segmentation_key() for s in MERGE_SPECS}
assert len({str(k) for k in keys.values()}) == 1, f"clés divergentes : {keys}"
print(f"{len(MERGE_SPECS)} conditions × {len(SEEDS)} seeds = {len(MERGE_SPECS) * len(SEEDS)} runs")
for s in MERGE_SPECS:
    sp = get_spec(s)
    merged = next((g.merge for g in sp.groups if g.merge), ())
    print(f"  {s:24s} input_dim={sp.input_dim}  fusion={merged or '—'}  hash={sp.hash()}")

5 conditions × 5 seeds = 25 runs
  v3_multi_novalid         input_dim=6  fusion=—  hash=77627d888210
  v3_merge_expressive      input_dim=6  fusion=('Expressive',)  hash=fa747af27cfc
  v3_merge_neither         input_dim=6  fusion=('Neither',)  hash=29fd51e5db82
  v3_merge_question        input_dim=6  fusion=('Question',)  hash=374ac6133558
  v3_merge_suggestion      input_dim=6  fusion=('Suggestion',)  hash=3213697d4296


In [4]:
# notebook — cellule 6 : exécution de la grille
rows = []
for spec_name in MERGE_SPECS:
    for seed in SEEDS:
        cfg = load_cfg(BASE_EXPERIMENT, seed=seed, spec_name=spec_name)
        art = train_gru(cfg)

        res = np.load(Path(RESULT_DIR) / art["hash"] / "gru_result.npz", allow_pickle=True)
        row = {"condition": spec_name, "seed": seed, "hash": art["hash"],
               "input_dim": art["feature_spec"]["input_dim"]}
        row.update(detection_metrics(res["y_true"], res["y_proba"], threshold=0.5))
        rows.append(row)
        print(f"--> {spec_name} seed {seed} : AUC {row['auc']:.4f}  "
              f"TPR@1%FPR {row['tpr@fpr1']:.4f}\n")

runs_merge = pd.DataFrame(rows)
out_dir = Path(RESULT_DIR) / "xai" / "merge_v3"
out_dir.mkdir(parents=True, exist_ok=True)
runs_merge.to_csv(out_dir / "merge_runs.csv", index=False)
print_table(runs_merge[["condition", "seed", "auc", "acc", "tpr@fpr1", "tpr@fpr0_1"]],
            title="Runs individuels", index=False)

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_multi_novalid
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
     

Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 10000, 'split': 'calibration', 'with_pdf': False, 'seed': 0}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_4fbe88291ebaf69cc7d4bd78214b8d51.pt
📦 Loading data from cache: /data2/cyaiche/cache_dir/results/implicite_segmentation/4fbe88291ebaf69cc7d4bd78214b8d51_eaf4e799/sentences_degrees_final.csv
appariement : 3017 paires | HWT écartés 0 | versions MGT écartées 1983
générateurs retenus : {'llama': 638, 'qwen': 627, 'gemini': 613, 'claude': 580, 'gpt4o': 559}
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'subset_size': 

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_multi_novalid
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
     


              precision    recall  f1-score   support

           0      0.891     0.867     0.879      4468
           1      0.870     0.894     0.882      4468

    accuracy                          0.881      8936
   macro avg      0.881     0.881     0.881      8936
weighted avg      0.881     0.881     0.881      8936

AUC globale : 0.9549

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.929598
      ICLR  2020  790 0.936619
   NeurIPS  2018  802 0.942612
   NeurIPS  2017  732 0.945997
   NeurIPS  2019  802 0.948589
      ICLR  2017  730 0.948801
      ICLR  2019  768 0.960381
      ICLR  2018  796 0.962242
   NeurIPS  2021 1070 0.981364
      ICLR  2021  976 0.986601

AUC par générateur :
 generator    n      auc
    llama 1767 0.922292
   gemini 1751 0.952415
     qwen 1853 0.961544
   claude 1780 0.964478
    gpt4o 1785 0.973316

 /data2/cyaiche/cache_dir/results/40267e07f7acff6aea5cee99ae7abf31/artifact.joblib
--> v3_multi_novalid seed 0 : AUC 0.954

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_multi_novalid
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
     


              precision    recall  f1-score   support

           0      0.931     0.842     0.884      4468
           1      0.856     0.937     0.895      4468

    accuracy                          0.890      8936
   macro avg      0.893     0.890     0.889      8936
weighted avg      0.893     0.890     0.889      8936

AUC globale : 0.9611

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.938509
      ICLR  2020  790 0.942189
   NeurIPS  2018  802 0.954627
      ICLR  2017  730 0.958003
   NeurIPS  2019  802 0.958390
   NeurIPS  2017  732 0.958845
      ICLR  2019  768 0.961005
      ICLR  2018  796 0.966251
   NeurIPS  2021 1070 0.982814
      ICLR  2021  976 0.990380

AUC par générateur :
 generator    n      auc
    llama 1767 0.922584
   gemini 1751 0.959891
     qwen 1853 0.969469
   claude 1780 0.973631
    gpt4o 1785 0.979826

 /data2/cyaiche/cache_dir/results/1ba399eda7c9218acf7ff6f7ee40770b/artifact.joblib
--> v3_multi_novalid seed 1 : AUC 0.961

... (40 outputs left out)

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_merge_suggestion
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
  


              precision    recall  f1-score   support

           0      0.820     0.843     0.831      4468
           1      0.838     0.815     0.826      4468

    accuracy                          0.829      8936
   macro avg      0.829     0.829     0.829      8936
weighted avg      0.829     0.829     0.829      8936

AUC globale : 0.9142

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.856719
   NeurIPS  2018  802 0.858036
   NeurIPS  2017  732 0.881282
      ICLR  2017  730 0.884429
   NeurIPS  2019  802 0.894970
      ICLR  2018  796 0.918380
      ICLR  2020  790 0.929063
      ICLR  2019  768 0.930196
   NeurIPS  2021 1070 0.969150
      ICLR  2021  976 0.984409

AUC par générateur :
 generator    n      auc
    llama 1767 0.884886
   gemini 1751 0.900258
     qwen 1853 0.905401
   claude 1780 0.937797
    gpt4o 1785 0.942496

 /data2/cyaiche/cache_dir/results/8b2c5cbcb41fde417419155bd3ff3b7c/artifact.joblib
--> v3_merge_suggestion seed 2 : AUC 0.

INFO:src.detector.subjectivity_aware_generation_estimator.training.gru_train:name: gru
seed: 42
hardware:
  device: cuda:0
dataset_val:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: test
  with_pdf: false
  seed: 0
dataset_train:
  name: aprdb
  description: AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt
  conference: null
  date: null
  keep_meta: true
  subset_size: 10000
  split: calibration
  with_pdf: false
  seed: 0
data_loader:
  batch_size: 1
  num_workers: 4
  pin_memory: true
training:
  split:
    strategy: predefined
  model:
    name: sage_gru_1
    method: training
    params_model:
      n_features: 6
      hidden: 32
      n_layers: 1
    params_training:
      epochs: 30
      lr: 0.001
    batch_size: 64
    feature_spec: v3_merge_suggestion
    feature_version:
      segmentation: 2
      subj_category:
      - Assertion
  


              precision    recall  f1-score   support

           0      0.825     0.856     0.840      4468
           1      0.851     0.819     0.834      4468

    accuracy                          0.838      8936
   macro avg      0.838     0.838     0.837      8936
weighted avg      0.838     0.838     0.837      8936

AUC globale : 0.9245

AUC par cellule :
 conference  year    n      auc
   NeurIPS  2016 1470 0.878462
   NeurIPS  2018  802 0.883129
   NeurIPS  2017  732 0.897966
   NeurIPS  2019  802 0.900971
      ICLR  2017  730 0.902083
      ICLR  2020  790 0.915046
      ICLR  2018  796 0.938947
      ICLR  2019  768 0.941149
   NeurIPS  2021 1070 0.972609
      ICLR  2021  976 0.984287

AUC par générateur :
 generator    n      auc
    llama 1767 0.886388
   gemini 1751 0.903241
     qwen 1853 0.924376
   claude 1780 0.952273
    gpt4o 1785 0.955245

 /data2/cyaiche/cache_dir/results/2285fd56da063759d6e6adda84e1b588/artifact.joblib
--> v3_merge_suggestion seed 3 : AUC 0.

In [5]:
# notebook — cellule 7 : importance LOFO par catégorie
REFERENCE = "v3_multi_novalid"
METRICS = ["auc", "acc", "tpr@fpr1", "tpr@fpr0_1"]

agg = runs_merge.groupby("condition")[METRICS].agg(["mean", "std"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg["n"] = runs_merge.groupby("condition").size()

ref = agg.loc[REFERENCE]
for m in METRICS:
    agg[f"delta_{m}"] = ref[f"{m}_mean"] - agg[f"{m}_mean"]
    agg[f"se_{m}"] = np.sqrt(agg[f"{m}_std"] ** 2 / agg["n"] + ref[f"{m}_std"] ** 2 / ref["n"])

agg["categorie"] = [c.replace("v3_merge_", "").capitalize() if c != REFERENCE else "—"
                    for c in agg.index]
# un delta sous 2 erreurs-types n'est pas interprétable
agg["significatif"] = np.where(agg["delta_auc"].abs() > 2 * agg["se_auc"], "oui", "non")

cols = ["categorie", "auc_mean", "auc_std", "delta_auc", "se_auc", "significatif",
        "delta_acc", "delta_tpr@fpr1", "se_tpr@fpr1", "delta_tpr@fpr0_1"]
print_table(agg[cols].sort_values("delta_auc", ascending=False),
            title="LOFO par fusion de catégorie — v3, 5 seeds")

== LOFO par fusion de catégorie — v3, 5 seeds ==
                      categorie  auc_mean  auc_std  delta_auc  se_auc significatif  delta_acc  delta_tpr@fpr1  se_tpr@fpr1  delta_tpr@fpr0_1
condition                                                                                                                                   
v3_merge_suggestion  Suggestion    0.9294   0.0108     0.0272  0.0050          oui     0.0376          0.1068       0.0167            0.0392
v3_merge_question      Question    0.9321   0.0041     0.0245  0.0022          oui     0.0353          0.0519       0.0068            0.0194
v3_merge_neither        Neither    0.9473   0.0044     0.0093  0.0023          oui     0.0125          0.0660       0.0187            0.0766
v3_merge_expressive  Expressive    0.9495   0.0060     0.0071  0.0030          oui     0.0109          0.0198       0.0129           -0.0460
v3_multi_novalid              —    0.9566   0.0027     0.0000  0.0017          non     0.0000          0.